# City General Hospital - 30-Day Readmission Prediction

This notebook demonstrates the full deep learning pipeline used in this repository.

- Load and inspect data
- Train a PyTorch MLP model
- Evaluate validation metrics
- Generate test predictions

In [1]:
from pathlib import Path
import json
import sys

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from train import train_model
from predict import infer

print('Repo root:', REPO_ROOT)

Repo root: /home/grow-lt-363/Desktop/DL_test/readmission-dl


In [2]:
TRAIN_PATH = REPO_ROOT / 'data' / 'train.csv'
TEST_PATH = REPO_ROOT / 'data' / 'test.csv'
ARTIFACTS_DIR = REPO_ROOT / 'artifacts'

print('Train exists:', TRAIN_PATH.exists())
print('Test exists:', TEST_PATH.exists())

Train exists: True
Test exists: True


In [6]:
model_path, config = train_model(
    train_path=TRAIN_PATH,
    artifacts_dir=ARTIFACTS_DIR,
    target_col=None,
)

print('Saved model:', model_path)
print('Validation metrics:', json.dumps(config['metrics'], indent=2))

Saved model: /home/grow-lt-363/Desktop/DL_test/readmission-dl/artifacts/model.pt
Validation metrics: {
  "roc_auc": 0.9483806528391703,
  "pr_auc": 0.7610033659501629,
  "f1": 0.7297297297297297,
  "precision": 0.675,
  "recall": 0.7941176470588235,
  "accuracy": 0.9473684210526315
}


In [7]:
PRED_PATH = REPO_ROOT / 'predictions.csv'
infer(
    input_csv=TEST_PATH,
    output_csv=PRED_PATH,
    artifacts_dir=ARTIFACTS_DIR,
)
print('Predictions saved:', PRED_PATH)

Saved predictions to: /home/grow-lt-363/Desktop/DL_test/readmission-dl/predictions.csv
Predictions saved: /home/grow-lt-363/Desktop/DL_test/readmission-dl/predictions.csv


In [9]:
# Final validation results in required format
with open(ARTIFACTS_DIR / 'config.json', 'r', encoding='utf-8') as f:
    artifact_cfg = json.load(f)

m = artifact_cfg.get('metrics', {})
threshold = float(artifact_cfg.get('threshold', 0.5))

# If some metrics are missing in config, compute them from saved model + validation split.
if any(k not in m for k in ['precision', 'recall', 'accuracy', 'f1', 'roc_auc']):
    import numpy as np
    import pandas as pd
    import torch
    import joblib
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, accuracy_score

    from train import to_binary_targets, detect_target_column, SEED
    from model import TabularReadmissionNet
    from preprocess import prepare_features

    train_df = pd.read_csv(TRAIN_PATH)
    target_col = artifact_cfg.get('target_col', detect_target_column(train_df))

    y = to_binary_targets(train_df[target_col])
    X = prepare_features(train_df.drop(columns=[target_col]))

    _, X_val_df, _, y_val = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
    )

    fitted = joblib.load(ARTIFACTS_DIR / 'preprocessor.joblib')
    X_val = fitted.transformer.transform(X_val_df)
    if hasattr(X_val, 'toarray'):
        X_val = X_val.toarray()
    X_val_t = torch.tensor(np.asarray(X_val), dtype=torch.float32)

    ckpt = torch.load(ARTIFACTS_DIR / 'model.pt', map_location='cpu')
    model = TabularReadmissionNet(
        input_dim=ckpt['input_dim'],
        hidden_dims=tuple(ckpt.get('hidden_dims', [128, 64])),
        dropout=float(ckpt.get('dropout', 0.2)),
    )
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()

    with torch.no_grad():
        val_prob = torch.sigmoid(model(X_val_t)).cpu().numpy()
    val_pred = (val_prob >= threshold).astype(int)

    m['roc_auc'] = float(roc_auc_score(y_val, val_prob))
    m['f1'] = float(f1_score(y_val, val_pred, zero_division=0))
    m['precision'] = float(precision_score(y_val, val_pred, zero_division=0))
    m['recall'] = float(recall_score(y_val, val_pred, zero_division=0))
    m['accuracy'] = float(accuracy_score(y_val, val_pred))


def _fmt(v):
    return 'N/A' if v is None else f"{float(v):.4f}"

print('Results on validation set')
print('Metric\tValue')
print(f"AUROC\t{_fmt(m.get('roc_auc'))}")
print(f"F1 (minority class)\t{_fmt(m.get('f1'))}")
print(f"Precision (minority)\t{_fmt(m.get('precision'))}")
print(f"Recall (minority)\t{_fmt(m.get('recall'))}")
print(f"Accuracy\t{_fmt(m.get('accuracy'))}")
print(f"Decision threshold used\t{threshold:.2f}")

Results on validation set
Metric	Value
AUROC	0.9484
F1 (minority class)	0.7297
Precision (minority)	0.6750
Recall (minority)	0.7941
Accuracy	0.9474
Decision threshold used	0.79
